# Chatbot Development Notebook

This notebook walks through the design of the course scheduling decision model for the MIT Course Advisor chatbot. We start by setting up a test student, define the factors that matter when choosing courses, compute those factors from real data, and then compare different model structures for combining them into course recommendations.

In [1]:
# ── Setup ──
import json, os
import numpy as np
from huggingface_hub import login, InferenceClient

from src.config import BASE_MODEL, MY_MODEL, HF_TOKEN, DATA_DIR
from src.chat import StudentProfile, get_feasible_candidates
from requirements import RequirementsTracker
from planner import SemesterPlanner
from scoring import (
    CourseScorer, FRAMEWORK_PROFILES, FACTORS, FACTORS_BY_DIM, DIMENSIONS,
    DEFAULT_SUB_WEIGHTS, score_linear, score_multidimensional, score_active_only, score_topk,
    compute_course_factors, fill_interest_defaults
)

login()
client = InferenceClient(model=MY_MODEL or BASE_MODEL, token=HF_TOKEN)

with open(os.path.join(DATA_DIR, "courses.json"), "r") as f:
    COURSES = {c["subject_id"]: c for c in json.load(f)}

tracker = RequirementsTracker(data_dir=DATA_DIR)
planner = SemesterPlanner(data_dir=DATA_DIR)

print(f"Model: {MY_MODEL or BASE_MODEL}")
print(f"Courses loaded: {len(COURSES)}")

Model: meta-llama/Llama-3.1-8B-Instruct
Courses loaded: 6808


---
# 1. Test Student Profile

All experiments in this notebook use the same test student so results are comparable across sections. We use a **6-4 (AI & Decision Making) sophomore** with 5 semesters remaining, planning for next fall. This student has completed first-year GIRs and introductory programming but hasn't started the EECS core or major-specific requirements.

We choose a less-advanced student deliberately: a student further along in their degree has fewer remaining requirements and fewer prerequisite chains, which makes the necessity factors less interesting. A sophomore still has critical path bottlenecks, requirement breadth to cover, and real tradeoffs between foundation-building and exploration.

In [2]:
profile = StudentProfile()
profile.major_id = "6-4"
profile.courses_taken = [
    "18.01", "18.02", "8.01", "8.02", "5.111", "7.012",  # GIR sciences
    "6.100A", "6.100B",  # intro programming
]
profile.semesters_left = 5
profile.next_is_fall = True
profile.max_units = 48

print("STUDENT PROFILE")
print("=" * 60)
print(f"Major: {profile.major_id}")
print(f"Year: Sophomore, {profile.semesters_left} semesters remaining")
print(f"Planning for: {'Fall' if profile.next_is_fall else 'Spring'}")
print(f"Courses completed: {len(profile.courses_taken)}")
print(f"Max units/semester: {profile.max_units}")
print()
print("GROUND TRUTH REQUIREMENTS")
print("=" * 60)
print(profile.get_ground_truth())

STUDENT PROFILE
Major: 6-4
Year: Sophomore, 5 semesters remaining
Planning for: Fall
Courses completed: 8
Max units/semester: 48

GROUND TRUTH REQUIREMENTS
GROUND TRUTH for 6-4 (Artificial Intelligence and Decision Making):
Courses taken: 18.01, 18.02, 8.01, 8.02, 5.111, 7.012, 6.100A, 6.100B
Required courses REMAINING: 6.1010, 6.1200, 6.1210, 18.C06
  ✓ Introductory Programming: complete (6.100A)
  ✗ Linear Algebra: need 1 more from: 18.C06, 18.06
  ✗ Probability/Inference: need 1 more from: 6.3700, 6.3800, 18.05
  ✗ Centers - Data-centric: need 1 more from: 6.3720, 6.3900, 6.C01
  ✗ Centers - Model-centric: need 1 more from: 6.3000, 6.3100, 6.4110, 6.4420, 6.4400
  ✗ Centers - Decision-centric: need 1 more from: 6.3100, 6.4110, 6.C571
  ✗ Centers - Computation-centric: need 1 more from: 6.1220, 6.1400, 6.4400, 6.C571
  ✗ Centers - Human-centric: need 1 more from: 6.3260, 6.3950, 6.4120, 6.4590, 6.C35
  ✗ Application CI-M: need 1 more from: 6.4200, 6.4210, 6.8611
  ✗ AI+D Advanced Und

---
# 2. Decision Factors

When recommending courses for a student's semester, the chatbot needs to evaluate each candidate along multiple dimensions. We identified 11 factors that matter when choosing a course, organized into three higher-level dimensions: **Necessity**, **Interest**, and **Feasibility**.

## 2.1 Necessity

Necessity captures how much the student *needs* this course to graduate on time. A course with high necessity is one the student can't afford to skip or defer without consequences.

**Critical path urgency**: Some courses are prerequisites for other required courses. If a student hasn't taken course A, and course B requires A, and course C requires B, then A sits at the start of a three-semester prerequisite chain. Deferring A delays B and C, potentially pushing graduation back. A course that unlocks many downstream requirements is more urgent than one with no dependents. We score this as 1.0 if the course unlocks 2+ downstream courses, 0.5 if it unlocks 1, and 0.0 if it unlocks none.

**Offering scarcity**: MIT courses are offered in fall, spring, or both. A course offered only in fall is scarce during a fall planning session — if the student doesn't take it now, they must wait a full year for the next opportunity. A course offered both semesters can be deferred to spring with no cost. Scored as 1.0 if only offered this semester, 0.0 if offered both.

**Requirement efficiency**: Some courses satisfy multiple requirement groups simultaneously. For example, in the 6-4 major, 6.4110 can count toward either the Model-centric or Decision-centric center requirement — taking it fills one requirement while preserving flexibility for the other. A course that double-counts across groups is more efficient than one that fills a single slot. Scored as 1.0 if the course appears in 2+ requirement groups, 0.0 otherwise.

**CI-M need**: MIT requires students to complete a certain number of Communication Intensive in the Major (CI-M) courses. If the student still needs CI-M credits and this course is designated CI-M, that's an additional reason to take it. Scored as 1.0 if the student needs CI-M and this course is CI-M, 0.0 otherwise.

## 2.2 Interest

Interest captures how much the student *wants* this course based on their personal goals and preferences. Unlike Necessity and Feasibility, these factors cannot be computed from catalog data — they require understanding the student's natural language preferences, which is where the LLM comes in.

**Interest match**: How well does the course topic align with the student's stated interests? If a student says "I'm interested in machine learning and NLP," a course on natural language processing scores high while a course on information policy scores low. The LLM reads the course description and the student's stated interests and judges the alignment on a 0-1 scale.

**Career relevance**: Does this course build toward the student's career or graduate school goals? A student targeting ML PhD programs benefits from taking advanced ML courses that signal depth to admissions committees. A student planning to enter product management benefits from courses with design and communication components. The LLM interprets the student's stated career direction and judges how much each course contributes.

**Learning style fit**: Does the course format match how the student prefers to learn? Some students thrive in project-based courses with hands-on implementation, others prefer lecture-based courses with problem sets, and others learn best through lab work and experimentation. The LLM matches the course format (inferred from the description) against the student's stated preferences.

## 2.3 Feasibility

Feasibility captures whether the student can realistically handle this course alongside everything else in their semester.

**Workload (absolute)**: How heavy is this course on its own, measured by in-class hours plus out-of-class hours per week? A 15-unit course with extensive projects is heavier than a 12-unit lecture course. We invert this so that lighter courses score higher (more feasible). Scored on a 0-1 scale where 0 is very heavy and 1 is light.

**Workload (relative)**: Absolute workload doesn't tell the full story — what matters is how this course fits with the *other* courses already planned. Adding a 15-unit project course when the student already has two other project courses is different from adding it alongside lighter lecture courses. We compute remaining unit capacity after adding this course and normalize: if the student still has plenty of room, the score is high; if they'd be at the cap, the score is low.

**Rating**: Student satisfaction rating from course evaluations (via FireRoad). A higher-rated course is more likely to be a well-taught, positive experience, which indirectly affects feasibility — a poorly rated course can be harder to get through regardless of the workload. FireRoad uses a 7-point scale; we normalize the typical 4–7 range to 0–1 via `(rating - 4) / 3`.

## 2.4 Computing Factor Values

We now compute the actual factor values for a diverse set of candidate courses. Rather than scoring only the student's feasible candidates (which are pre-filtered to be similar), we deliberately include courses that test the model's ability to differentiate:

| Course | Type | Why it's in the test set |
|--------|------|-------------------------|
| 6.1200 (Math for CS) | Good fit | On the critical path — unlocks a 3-semester chain (6.1200 → 6.1210 → 6.1220) |
| 6.1010 (Fundamentals of Programming) | Good fit | Foundation course, unlocks 6.4200 and others |
| 18.C06 (Linear Algebra) | Good fit | Fills required linear algebra slot, unlocks center courses |
| 6.3900 (Intro to ML) | Too advanced | Requires 6.1010 + 18.C06, which the student hasn't taken |
| 14.01 (Microeconomics) | HASS | Counts toward HASS GIR requirement, not a major course |
| 2.74 (Bio-Inspired Robotics) | Random | Outside the major entirely — shouldn't rank high |

Python computes the Necessity and Feasibility factors by calling `compute_course_factors()`, which queries the requirement tracker and prerequisite graph. Interest factors are set manually for now (the LLM will score these in later experiments).

In [4]:
# ── Compute factor values for our diverse test set ──

test_courses = ["6.1200", "6.1010", "18.C06", "6.3900", "14.01", "2.74"]

# Python computes necessity + feasibility from catalog data and student profile
candidate_data = compute_course_factors(
    test_courses, profile, COURSES, tracker, planner,
    max_units=profile.max_units, planned_units=0
)

# Set interest factors manually for our ML/NLP-interested student
# (In later experiments, the LLM will score these from course descriptions)
interest_values = {
    "6.1200": {"interest_match": 0.3, "career_relevance": 0.4, "learning_style_fit": 0.5},
    "6.1010": {"interest_match": 0.5, "career_relevance": 0.5, "learning_style_fit": 0.7},
    "18.C06": {"interest_match": 0.4, "career_relevance": 0.5, "learning_style_fit": 0.4},
    "6.3900": {"interest_match": 1.0, "career_relevance": 0.9, "learning_style_fit": 0.8},
    "14.01":  {"interest_match": 0.3, "career_relevance": 0.2, "learning_style_fit": 0.5},
    "2.74":   {"interest_match": 0.4, "career_relevance": 0.1, "learning_style_fit": 0.6},
}

# Merge: Python factors + manual interest
candidates = {}
for cid in test_courses:
    d = candidate_data[cid]
    factors = dict(d["factors"])
    for k, v in interest_values.get(cid, {}).items():
        factors[k] = v
    candidates[cid] = factors

# Display
print("FACTOR VALUES")
print("=" * (30 + 9 * len(test_courses)))
print(f"{'':30s}", end="")
for cid in test_courses:
    print(f" {cid:>8s}", end="")
print()
print("-" * (30 + 9 * len(test_courses)))

for dim in DIMENSIONS:
    print(f"  [{dim}]")
    for f in FACTORS_BY_DIM[dim]:
        print(f"    {f:26s}", end="")
        for cid in test_courses:
            val = candidates[cid].get(f, 0)
            if val is None:
                print(f" {'—':>8s}", end="")
            elif val >= 0.7:
                print(f" {val:>7.2f}*", end="")
            else:
                print(f" {val:>8.2f}", end="")
        print()

print(f"\n  * = 0.7 or above")

# Show flags (prereqs not met, etc.)
print(f"\nFLAGS:")
for cid in test_courses:
    d = candidate_data[cid]
    title = d['title'][:35]
    req = d['requirement_filled']
    flags = d.get('flags', [])
    flag_str = ", ".join(flags) if flags else "(none)"
    print(f"  {cid:>8s}  {title:<35s}  req={req:<25s}  {flag_str}")

FACTOR VALUES
                                 6.1200   6.1010   18.C06   6.3900    14.01     2.74
------------------------------------------------------------------------------------
  [necessity]
    critical_path                 1.00*     0.00     0.50     0.00     0.00     0.00
    scarcity                       0.00     0.00    1.00*     0.00     0.00    1.00*
    efficiency                     0.00     0.00     0.00     0.00     0.00     0.00
    ci_m_need                      0.00     0.00     0.00     0.00     0.00     0.00
  [interest]
    interest_match                 0.30     0.50     0.40    1.00*     0.30     0.40
    career_relevance               0.40     0.50     0.50    0.90*     0.20     0.10
    learning_style_fit             0.50    0.70*     0.40    0.80*     0.50     0.60
  [feasibility]
    workload_absolute              0.57     0.56     0.56     0.68    0.76*     0.41
    rating                         0.52     0.66     0.54     0.50     0.25    0.77*

  * = 0

**Observations:**

The factor values reveal a few patterns. Critical path correctly identifies 6.1200 as the highest-priority course (it unlocks a 3-semester chain), with 18.C06 at medium priority. Scarcity flags 18.C06 and 2.74 as fall-only, though for 2.74 this is misleading — it's outside the major, so the urgency is artificial. Efficiency is 0.0 across the board because this student is too early in the major for double-counting to kick in. The interest factors (manually set here) do their job separating 6.3900 as the strongest match from less relevant options like 14.01 and 2.74.

---
# 3. Decision Model Structure

With the factors defined and computed, we now need a model that combines them into a single score per course. The choice of model structure determines how factors interact, how many parameters the LLM needs to adjust to personalize recommendations, and how the system handles missing information.

We consider four approaches.

## 3.1 Approach A: Flat Linear Model

The simplest approach: weight all 11 factors independently and sum them.

$$\text{score} = \frac{\sum_{i=1}^{11} w_i \cdot f_i}{\sum_{i=1}^{11} w_i}$$

where $f_i \in [0, 1]$ is the normalized factor value and $w_i$ is its weight. Each factor gets its own weight, and the final score is the weighted average.

**Advantages**: Simple and fully transparent — you can trace exactly why a course scored the way it did by looking at which factors contributed and how much.

**Disadvantages**: Two problems emerge. First, **low-priority factors add noise**: a course that scores modestly on 8 irrelevant factors can outscore one that is very strong on the 2 factors the student actually cares about. The noise floor is high because every factor contributes. Second, the model **doesn't reflect the dimensional structure** among the factors. Necessity has 5 sub-factors while Interest has 3 — in a flat model, necessity-related concerns get 5/11 of the total weight just by headcount, even if the student cares equally about necessity and interest as overall categories. Personalizing the model also requires the LLM to adjust 11 individual weights, which is error-prone for a smaller language model.

## 3.2 Approach B: Multi-Dimensional Model

To address the structural issues in the linear model, we first combine factors within each dimension, then weight the three dimensions at the top level:

$$\text{necessity} = \frac{\sum_{j \in \text{nec}} \alpha_j \cdot f_j}{\sum_{j \in \text{nec}} \alpha_j} \qquad \text{interest} = \frac{\sum_{j \in \text{int}} \alpha_j \cdot f_j}{\sum_{j \in \text{int}} \alpha_j} \qquad \text{feasibility} = \frac{\sum_{j \in \text{feas}} \alpha_j \cdot f_j}{\sum_{j \in \text{feas}} \alpha_j}$$

$$\text{score} = W_1 \cdot \text{necessity} + W_2 \cdot \text{interest} + W_3 \cdot \text{feasibility}$$

The sub-factor weights $\alpha_j$ within each dimension are fixed and reflect the relative importance of each sub-factor (e.g., critical path is weighted higher than CI-M need within the Necessity dimension). The LLM adjusts only the 3 top-level dimension weights $(W_1, W_2, W_3)$ to personalize the model.

The sub-factor weights within each dimension are:

| Dimension | Factor | Weight | Rationale |
|-----------|--------|--------|----------|
| **Necessity** | critical_path | 3.0 | Most consequential — delays entire graduation |
| | scarcity | 2.5 | Real cost to waiting a full year |
| | efficiency | 2.0 | Useful optimization but not urgent |
| | ci_m_need | 1.0 | Binary check, least differentiating |
| **Interest** | interest_match | 3.0 | Primary driver of preference |
| | career_relevance | 2.0 | Important but more speculative |
| | learning_style_fit | 1.0 | Secondary, most students adapt |
| **Feasibility** | workload_relative | 3.0 | Most contextual to this specific semester |
| | workload_absolute | 2.0 | Matters but less contextual |
| | rating | 1.5 | Useful signal, shouldn't dominate |

**Advantages**: The LLM controls 3 knobs instead of 11, which is far more reliable for a smaller model. The dimensional structure maps to how students actually think about course selection: "do I need this, do I want this, can I handle this." Each dimension produces an interpretable sub-score that the chatbot can explain. When the student's interests are unknown (early in a conversation), we can set $W_2 = 0$ to effectively ignore interest — giving us active-only behavior without switching models.

**Disadvantages**: Assumes the sub-factor weights within each dimension are correct. If critical path urgency should matter much more than CI-M need in some situations, the fixed sub-weights can't adapt to that. However, this could be addressed by adjusting normalization ranges rather than adding another layer of tunable weights.

## 3.3 Approach C: Active-Only Model

Instead of always using all three dimensions, only the most relevant dimensions are active. Inactive dimensions are zeroed out before scoring:

$$\text{score} = \frac{\sum_{d \in \text{active}} W_d \cdot \text{dim}_d}{\sum_{d \in \text{active}} W_d}$$

The set of active dimensions is determined by what information is available. Early in a conversation when only the student's profile is known, only Necessity and Feasibility are active (since Python can compute these without any conversation). Once the student states their interests, Interest activates and the ranking updates.

**Advantages**: Handles incomplete information gracefully. Scoring only on known dimensions avoids the problem of defaulting unknown interest values to 0.5 (which could mislead the ranking by treating all courses as equally interesting). Eliminates noise from dimensions the student hasn't expressed any preference about.

**Disadvantages**: Requires a mechanism to decide when dimensions activate. The ranking can shift abruptly when a new dimension turns on, which might confuse the student if the chatbot's recommendations suddenly change. Less interpretable when the active set varies across turns.


## 3.4 Approach D: Top-K Sub-Factor Model

Like the multidimensional model, this approach aggregates within dimensions then combines across them. The difference is that within each dimension, only the **k highest-valued sub-factors** contribute to the dimension score — lower-valued sub-factors are zeroed out.

For example, if a course has `critical_path=1.0` but `scarcity=0.0`, `efficiency=0.0`, `ci_m_need=0.0`, the standard multidimensional model averages these, diluting the strong critical path signal. With $k=1$, the necessity score would be 1.0 (purely from critical path). With $k=2$, the two highest values contribute.

$$D_{\text{dim}} = \frac{\sum_{i \in \text{top-}k} \alpha_i \cdot f_i}{\sum_{i \in \text{top-}k} \alpha_i}$$

**Advantages**: Prevents weak sub-factors from diluting a strong signal. A course that is extremely urgent on one axis (e.g., only offered this semester) doesn't get penalized for having zero efficiency or zero CI-M need.

**Disadvantages**: Introduces another hyperparameter ($k$). At $k=1$, a dimension's score is entirely determined by a single sub-factor, which may be too aggressive. At $k$ equal to the number of sub-factors, it reduces to the standard multidimensional model.

## 3.5 Experiment: Baseline Comparison

We score all candidates under each model structure with balanced dimension weights. Do the three structures agree on the ranking? Where do they disagree, and why?

In [5]:
# ── Experiment 1: Baseline comparison ──

print("EXPERIMENT 1: Model Structure Comparison (balanced weights)")
print("=" * 80)

results_by_model = {}
for model_type in ["linear", "multidimensional", "active_only", "topk"]:
    scorer = CourseScorer(model=model_type)
    results = scorer.score_candidates(candidates)
    results_by_model[model_type] = results

# Show full ranking for each model
for model_type, results in results_by_model.items():
    print(f"\n  {model_type.upper()}:")
    for rank, (cid, score, details) in enumerate(results, 1):
        title = candidate_data[cid]["title"][:35]
        flags = candidate_data[cid].get("flags", [])
        flag_str = "  ⚠ " + ", ".join(flags) if flags else ""
        print(f"    #{rank}  {score:.3f}  {cid:8s}  {title}{flag_str}")

# Agreement check
top3 = {m: [cid for cid, _, _ in r[:3]] for m, r in results_by_model.items()}
print(f"\n  Top-3 agreement: {top3['linear'] == top3['multidimensional'] == top3['active_only'] == top3['topk']}")

# Dimensional breakdown (multi-dim)
print(f"\n  Dimensional Breakdown (multi-dimensional, balanced):")
print(f"  {'Course':>8s}  {'Score':>6s}  {'Necess':>7s}  {'Intere':>7s}  {'Feasib':>7s}  Strongest")
print(f"  {'-'*60}")
for cid, score, details in results_by_model["multidimensional"]:
    ds = details.get("dim_scores", {})
    strongest = max(ds, key=ds.get) if ds else "?"
    print(f"  {cid:>8s}  {score:6.3f}  {ds.get('necessity',0):7.3f}  {ds.get('interest',0):7.3f}  {ds.get('feasibility',0):7.3f}  ← {strongest}")

EXPERIMENT 1: Model Structure Comparison (balanced weights)

  LINEAR:
    #1  0.477  18.C06    Linear Algebra and Optimization
    #2  0.437  6.3900    Introduction to Machine Learning  ⚠ PREREQS_NOT_MET: 6.1010 or 6.1210, 18.03 or 18.06 or 18.700 or 18.C06
    #3  0.402  6.1200    Mathematics for Computer Science
    #4  0.366  2.74      Bio-inspired Robotics
    #5  0.308  6.1010    Fundamentals of Programming
    #6  0.223  14.01     Principles of Microeconomics

  MULTIDIMENSIONAL:
    #1  0.510  6.3900    Introduction to Machine Learning  ⚠ PREREQS_NOT_MET: 6.1010 or 6.1210, 18.03 or 18.06 or 18.700 or 18.C06
    #2  0.481  18.C06    Linear Algebra and Optimization
    #3  0.417  6.1200    Mathematics for Computer Science
    #4  0.385  2.74      Bio-inspired Robotics
    #5  0.367  6.1010    Fundamentals of Programming
    #6  0.272  14.01     Principles of Microeconomics

  ACTIVE_ONLY:
    #1  0.510  6.3900    Introduction to Machine Learning  ⚠ PREREQS_NOT_MET: 6.1010 or 6.12

**Experiment 1 Discussion:**

The four models agree on the bottom two (6.1010, 14.01) but diverge at the top. Linear ranks 18.C06 first because its scarcity score (fall-only offering) gets full weight alongside all other factors, pushing it above 6.3900 whose strong interest scores are diluted across the flat average. Multidimensional and active-only (identical here since all dimensions are active) flip this: 6.3900's dominant interest dimension (0.93) is preserved by the within-dimension aggregation, so it outranks 18.C06 despite lower necessity. Top-K (k=2) puts 18.C06 back on top — by keeping only the 2 strongest sub-factors per dimension, it amplifies 18.C06's scarcity signal within necessity while also compressing the score range (top-4 are all within 0.09 of each other), making the rankings more fragile.
The key structural difference: linear lets individual sub-factors bleed across dimensions (scarcity inflates 2.74, an irrelevant out-of-major course), multidimensional contains this by normalizing within dimensions first, and top-K concentrates each dimension's score on peak signals at the cost of tighter score separation.

## 3.5 Experiment: Framework Stress Test

The model structure only matters if different weights produce different rankings. We test four student archetypes, each with different dimension weights:

| Framework | Necessity | Interest | Feasibility | Description |
|-----------|-----------|----------|-------------|-------------|
| Requirements Optimizer | 0.60 | 0.10 | 0.30 | Graduate as efficiently as possible |
| Interest Explorer | 0.15 | 0.55 | 0.30 | Take courses that sound interesting |
| Workload Balancer | 0.20 | 0.15 | 0.65 | Keep the semester manageable |
| Balanced | 0.35 | 0.35 | 0.30 | No single priority dominates |

For each framework and model structure, we check: does the top course change? If a model always recommends the same course regardless of weights, it isn't responsive enough to personalization.

In [6]:
# ── Experiment 2: Framework stress test ──

frameworks = ["requirements_optimizer", "interest_explorer", "workload_balancer", "balanced"]

print("EXPERIMENT 2: Framework Stress Test")
print("=" * 95)
print(f"\n{'Framework':<25s} {'Model':<15s} {'Weights (N/I/F)':<18s} {'#1':>8s} {'#2':>8s} {'#3':>8s}")
print("-" * 95)

top1_by_model = {m: {} for m in ["linear", "multidimensional", "active_only", "topk"]}

for fw_name in frameworks:
    for model_type in ["multidimensional", "linear", "active_only", "topk"]:
        scorer = CourseScorer(model=model_type)
        scorer.set_framework(fw_name)
        results = scorer.score_candidates(candidates)
        
        top1_by_model[model_type][fw_name] = results[0][0]
        w = scorer.dim_weights
        w_str = f"{w['necessity']:.2f}/{w['interest']:.2f}/{w['feasibility']:.2f}"
        r = [cid for cid, _, _ in results[:3]]
        
        print(f"{fw_name[:24]:<25s} {model_type:<15s} {w_str:<18s} {r[0]:>8s} {r[1]:>8s} {r[2]:>8s}")
    print()

# Summary
print("SENSITIVITY TO FRAMEWORK (distinct #1 courses across 4 frameworks):")
for m in ["linear", "multidimensional", "active_only", "topk"]:
    distinct = len(set(top1_by_model[m].values()))
    label = "HIGH" if distinct >= 3 else "MEDIUM" if distinct == 2 else "LOW"
    print(f"  {m:20s}: {distinct} distinct #1 courses  [{label}]")
    for fw, cid in top1_by_model[m].items():
        print(f"    {fw[:25]:25s} → {cid}")

EXPERIMENT 2: Framework Stress Test

Framework                 Model           Weights (N/I/F)          #1       #2       #3
-----------------------------------------------------------------------------------------------
requirements_optimizer    multidimensional 0.60/0.10/0.30       18.C06   6.1200     2.74
requirements_optimizer    linear          0.60/0.10/0.30       18.C06   6.1200     2.74
requirements_optimizer    active_only     0.60/0.10/0.30       18.C06   6.1200     2.74
requirements_optimizer    topk            0.60/0.10/0.30       18.C06   6.1200     2.74

interest_explorer         multidimensional 0.15/0.55/0.30       6.3900   18.C06   6.1010
interest_explorer         linear          0.15/0.55/0.30       6.3900   18.C06   6.1010
interest_explorer         active_only     0.15/0.55/0.30       6.3900   18.C06   6.1010
interest_explorer         topk            0.15/0.55/0.30       6.3900   18.C06   6.1200

workload_balancer         multidimensional 0.20/0.15/0.65       6.3900 

**Experiment 2 Discussion:**

All four models show the same top-level sensitivity (2 distinct #1 courses across 4 frameworks), so no model is strictly more responsive to framework changes. The meaningful split is in which frameworks produce which #1: multidimensional and active-only give 6.3900 the top slot in 3 of 4 frameworks (all except requirements_optimizer), while linear and top-K default to 18.C06 in 3 of 4 (only flipping for interest_explorer). This reflects the same structural pattern from Experiment 1 — linear and top-K let necessity-driven scores (18.C06's scarcity) dominate unless interest weight is explicitly high, while multidimensional's within-dimension normalization gives the interest dimension enough room to push 6.3900 ahead under balanced and workload_balancer weights. The workload_balancer row is the clearest discriminator: multidimensional correctly ranks 6.3900 first (it has the best feasibility profile), while linear and top-K keep 18.C06 on top despite feasibility being weighted at 0.65, because scarcity still bleeds through.

## 3.6 Experiment: Information Availability

Early in a conversation, the chatbot knows the student's major and courses taken (so Necessity is computable) but hasn't learned their interests. Should the model:
- Use all dimensions with Interest defaulted to 0.5? (guess)
- Use only Necessity? (don't guess)
- Use Necessity + Feasibility, ignoring Interest? (partial info)

We simulate information arriving over a conversation and check whether the ranking changes as dimensions activate.

In [7]:
# ── Experiment 3: Information availability ──

# Create a neutral-interest version (all interest = 0.5)
candidates_neutral = {}
for cid, factors in candidates.items():
    f = dict(factors)
    f["interest_match"] = 0.5
    f["career_relevance"] = 0.5
    f["learning_style_fit"] = 0.5
    candidates_neutral[cid] = f

print("EXPERIMENT 3: Information Availability")
print("=" * 85)
print("\nSimulating a conversation where information arrives over time.")
print("Student is interested in ML/NLP — but the chatbot doesn't know that yet.\n")

scenarios = [
    ("Stage 1: Necessity only",        "active_only", candidates,         ["necessity"]),
    ("Stage 2: Necessity + Feasib.",   "active_only", candidates,         ["necessity", "feasibility"]),
    ("Stage 3: All dims, interest=0.5","multidimensional", candidates_neutral, None),
    ("Stage 4: All dims, real interest","multidimensional", candidates,        None),
]

prev_top3 = None
for label, model, cands, active_dims in scenarios:
    scorer = CourseScorer(model=model)
    if active_dims:
        scorer.set_active_dims(active_dims)
    results = scorer.score_candidates(cands)
    
    top3 = [cid for cid, _, _ in results[:3]]
    ranking_str = " > ".join(f"{cid} ({score:.3f})" for cid, score, _ in results)
    changed = "" if prev_top3 is None else (" ← SAME" if top3 == prev_top3 else " ← CHANGED")
    
    print(f"  {label}")
    print(f"    {ranking_str}{changed}")
    print()
    prev_top3 = top3

EXPERIMENT 3: Information Availability

Simulating a conversation where information arrives over time.
Student is interested in ML/NLP — but the chatbot doesn't know that yet.

  Stage 1: Necessity only
    18.C06 (0.375) > 6.1200 (0.250) > 2.74 (0.250) > 6.1010 (0.000) > 6.3900 (0.000) > 14.01 (0.000)

  Stage 2: Necessity + Feasib.
    18.C06 (0.455) > 2.74 (0.406) > 6.1200 (0.386) > 6.1010 (0.283) > 6.3900 (0.273) > 14.01 (0.234) ← CHANGED

  Stage 3: All dims, interest=0.5
    18.C06 (0.505) > 6.1200 (0.464) > 2.74 (0.444) > 6.3900 (0.358) > 6.1010 (0.356) > 14.01 (0.342) ← CHANGED

  Stage 4: All dims, real interest
    6.3900 (0.510) > 18.C06 (0.481) > 6.1200 (0.417) > 2.74 (0.385) > 6.1010 (0.367) > 14.01 (0.272) ← CHANGED



**Experiment 3 Discussion:**

The progression validates the active-only design rationale: with only necessity available, the model sensibly ranks 18.C06 first (highest scarcity + critical path). Adding feasibility reshuffles the middle but keeps the same #1. The real shift happens at Stage 4, when actual interest scores arrive and 6.3900 jumps from 5th to 1st — exactly the behavior we want, since a student interested in ML/NLP should see that course surface once the chatbot learns their preferences. This demonstrates that the model degrades gracefully with active-only rather than producing garbage rankings, and that interest is the factor with the most power to reorder results.

## 3.7 Experiment: Sensitivity Analysis

How stable are the rankings under small weight perturbations? If the #1 course changes when we nudge weights by ±10%, the model is fragile and small errors in weight estimation lead to unreliable recommendations.

In [8]:
# ── Experiment 4: Sensitivity analysis ──

n_trials = 200
perturbation = 0.10
np.random.seed(42)

print("EXPERIMENT 4: Sensitivity Analysis")
print(f"±{perturbation*100:.0f}% perturbation on balanced weights, {n_trials} trials per model")
print("=" * 75)

for model_type in ["linear", "multidimensional", "active_only", "topk"]:
    top1_counts = {}
    top2_stable = 0
    base_scorer = CourseScorer(model=model_type)
    base_results = base_scorer.score_candidates(candidates)
    base_top2 = set(cid for cid, _, _ in base_results[:2])
    
    for _ in range(n_trials):
        perturbed = {}
        for dim in DIMENSIONS:
            base_w = {"necessity": 0.35, "interest": 0.35, "feasibility": 0.30}[dim]
            perturbed[dim] = max(0.01, base_w + np.random.uniform(-perturbation, perturbation))
        total = sum(perturbed.values())
        perturbed = {d: w/total for d, w in perturbed.items()}
        
        scorer = CourseScorer(model=model_type)
        scorer.dim_weights = perturbed
        results = scorer.score_candidates(candidates)
        
        top1 = results[0][0]
        top1_counts[top1] = top1_counts.get(top1, 0) + 1
        if set(cid for cid, _, _ in results[:2]) == base_top2:
            top2_stable += 1
    
    dominant_pct = max(top1_counts.values()) / n_trials * 100
    label = "STABLE" if dominant_pct > 80 else "MODERATE" if dominant_pct > 50 else "FRAGILE"
    
    print(f"\n  {model_type.upper():20s} [{label}]")
    print(f"    Top-1 consistency: {dominant_pct:.0f}%   Top-2 pair consistency: {top2_stable/n_trials*100:.0f}%")
    for cid in sorted(top1_counts, key=top1_counts.get, reverse=True):
        print(f"      {cid}: {top1_counts[cid]}/{n_trials} ({top1_counts[cid]/n_trials*100:.0f}%)")

EXPERIMENT 4: Sensitivity Analysis
±10% perturbation on balanced weights, 200 trials per model

  LINEAR               [MODERATE]
    Top-1 consistency: 80%   Top-2 pair consistency: 82%
      18.C06: 160/200 (80%)
      6.3900: 40/200 (20%)

  MULTIDIMENSIONAL     [MODERATE]
    Top-1 consistency: 70%   Top-2 pair consistency: 100%
      6.3900: 139/200 (70%)
      18.C06: 61/200 (30%)

  ACTIVE_ONLY          [MODERATE]
    Top-1 consistency: 76%   Top-2 pair consistency: 100%
      6.3900: 153/200 (76%)
      18.C06: 47/200 (24%)

  TOPK                 [STABLE]
    Top-1 consistency: 81%   Top-2 pair consistency: 60%
      18.C06: 162/200 (81%)
      6.3900: 38/200 (19%)


**Experiment 4 Discussion:**

All four models are moderate-to-stable on top-1 consistency (70–81%), but they diverge sharply on top-2 pair consistency. Multidimensional and active-only maintain 100% top-2 pair consistency — 6.3900 and 18.C06 always occupy the top two slots, they just swap positions. This means small weight perturbations change the #1 pick but never let a third course sneak into the top 2, which is a desirable property: the model is sensitive enough that dimension weights matter, but stable enough that the recommendation set is reliable. Top-K has the highest top-1 consistency (81%) but the lowest top-2 pair consistency (60%) — under perturbation, other courses break into the top 2, confirming the compressed score separation observed in Experiment 1. Linear sits in between (82% pair consistency) but defaults to 18.C06 in 80% of trials, reinforcing the pattern that linear is harder for dimension weights to steer away from necessity-dominated rankings. Multidimensional's 70% top-1 is the lowest, but paired with 100% top-2 consistency this is the best profile for LLM-controlled weights: adjustments reliably change the #1 pick without destabilizing the overall recommendation set.

## 3.8 Model Selection

Based on the four experiments, we select the model structure the chatbot will use.

**Model Selection:**

1. **FRAMEWORK RESPONSIVENESS (Exp 2):** All four models show the same top-level sensitivity (2 distinct #1 courses across 4 frameworks), but they differ in which frameworks they respond to. Multidimensional and active-only give 6.3900 the #1 slot in 3 of 4 frameworks, while linear and top-K default to 18.C06 in 3 of 4. This is because linear and top-K let necessity-driven scores (18.C06's scarcity) dominate unless interest weight is explicitly high, while multidimensional's within-dimension normalization gives the interest dimension enough room to steer rankings under balanced and feasibility-heavy weights. The workload_balancer row is the clearest discriminator: multidimensional correctly ranks 6.3900 first (best feasibility profile) even at 0.65 feasibility weight, while linear and top-K keep 18.C06 on top because scarcity bleeds through.

2. **STABILITY (Exp 4):** Multidimensional has the lowest top-1 consistency (70%) but maintains 100% top-2 pair consistency — 6.3900 and 18.C06 always occupy the top two slots, they just swap positions under perturbation. This is the ideal profile for LLM-controlled weights: adjustments reliably change the #1 pick without destabilizing the overall recommendation set. Linear is more stable at the top (80% top-1) but harder to steer, and top-K has the highest top-1 consistency (81%) but the lowest top-2 pair consistency (60%) — other courses break into the top 2 under perturbation, reflecting the compressed score separation observed in Experiment 1.

3. **INFORMATION HANDLING (Exp 3)**: Scoring on partial information works well. With only necessity available, 18.C06 correctly ranks first. Adding feasibility reshuffles the middle without misleading. The real validation is Stage 4: when actual interest scores arrive, 6.3900 jumps from #4 to #1, which is exactly the right behavior. Active-only's dimension-zeroing can be replicated within multidimensional by setting dimension weights to zero, so a separate model is unnecessary.

4. **INTERPRETABILITY (Exp 1):** Multidimensional is the most interpretable. The dimensional breakdown gives a clear explanation: "6.3900 ranks first because it dominates on interest (0.93) despite zero necessity." Linear flattens this into a single number with no natural explanation structure. Top-K compresses score separation (top-4 within 0.09), making rankings harder to explain and more sensitive to noise.

### Selected Model: Multidimensional with Framework Profiles

We select the **multidimensional model** (Approach B). The scoring is a two-stage aggregation:

**Stage 1 — Within each dimension,** sub-factors are combined using sub-factor weights $\alpha_i$:

$$D_{\text{dim}} = \frac{\sum_{i \in \text{dim}} \alpha_i \cdot f_i}{\sum_{i \in \text{dim}} \alpha_i}$$

**Stage 2 — Across dimensions,** dimension scores are combined using dimension weights:

$$\text{score} = W_N \cdot D_N + W_I \cdot D_I + W_F \cdot D_F$$

This gives the LLM a **two-level control hierarchy:**

| Control level | What it sets | How it's set |
|--------------|-------------|-------------|
| **Dimension weights** $(W_N, W_I, W_F)$ | The balance between necessity, interest, and feasibility | LLM adjusts per-turn based on student messages (tested in Section 5) |
| **Sub-factor weights** $\alpha_i$ | The relative importance of factors *within* a dimension | Set by **framework profiles** — predefined weight vectors for common student archetypes |

Framework profiles capture recurring patterns in how sub-factors should be weighted. For example:

| Profile | Sub-factor override | Effect |
|---------|-------------------|--------|
| `career_strategist` | `career_relevance`: 2.0, `interest_match`: 1.0, `learning_style_fit`: 0.5 | Within interest, career alignment dominates over topic curiosity |
| `balanced` | (default $\alpha_i$) | No overrides — sub-factors weighted by design defaults |

The key reason to choose multidimensional over linear: **the LLM is actively controlling dimension weights.** Experiment 4 showed multidimensional is more sensitive to dimension weight changes (70% top-1 stability vs linear's 90%). Under random perturbation that's instability, but under intentional LLM control it means the student's preferences have more impact on the final ranking. Linear's stability would dampen the LLM's adjustments.

→ **LLM's job:** (1) Determine which dimension weights to adjust based on the student's messages. (2) Score the 3 interest factors that Python cannot compute from catalog data.

→ **Python's job:** Compute the 6 factual factors, aggregate within dimensions using sub-factor weights (default or framework-overridden), then combine dimensions using LLM-adjusted weights.

The open questions — tested in Sections 4 and 5 — are whether the LLM can reliably detect *which* dimensions to adjust (Section 4) and *how* it should adjust them (Section 5).

---
# 4. LLM Dimension Detection

The selected model (Section 3.8) gives the LLM control over the dimension weights $(W_N, W_I, W_F)$. But before we can test *how* the LLM adjusts those weights, we need to answer a more basic question: **can the LLM correctly identify which dimension(s) a student's message is about?**

This is the signal extraction problem. A student says something in natural language; the system must determine:

1. **Which dimension(s) are relevant** — is the student talking about graduation requirements (necessity), personal interests (interest), or workload constraints (feasibility)?
2. **What direction** — should that dimension's weight go up (BOOST) or down (REDUCE)?
3. **At what granularity** — is the student expressing a dimension-level preference ("I care about workload") or targeting a specific sub-factor ("I want highly-rated courses")?

This is hard because:
- A single message can contain **multiple signals** at different dimensions ("I want interesting courses but also need to graduate")
- Some signals are **explicit** ("I care about workload") and some are **implicit** (the student keeps asking about hours per week)
- Some messages are **ambiguous** — "I want to take 6.3900" could be interest-driven or necessity-driven depending on context
- The system must distinguish **new information** from reinforcement of existing preferences

We test this with a labeled battery of student messages where the ground truth dimension labels are set by human judgment. We compare two approaches:

| Approach | Mechanism | Pros | Cons |
|----------|-----------|------|------|
| **Python regex** (`detect_signals`) | Pattern matching on keywords/phrases | Fast, deterministic, no API cost | Misses paraphrases, no context understanding |
| **LLM classification** | Prompt the LLM to output structured signals | Understands paraphrase, context, nuance | Slower, may produce malformed output, costs an API call |

The experiments below test both approaches on the same battery, then compare accuracy.

## 4.1 Ground Truth Test Battery

We define a set of student messages with human-labeled ground truth signals. Each message has:
- **Expected dimension signals:** which dimensions should be BOOSTed or REDUCEd
- **Difficulty level:** easy (single explicit signal), medium (multi-signal or implicit), hard (ambiguous or requires context)

This battery is used by both the regex baseline (Experiment 5) and the LLM classifier (Experiment 6).

In [ ]:
# ── Ground truth test battery ──
# Each entry: (message, expected_signals, difficulty, notes)
# expected_signals: dict of {dimension: "BOOST" or "REDUCE"}

TEST_BATTERY = [
    # ── Easy: single explicit signal ──
    (
        "I need to graduate on time, what requirements am I missing?",
        {"necessity": "BOOST"},
        "easy", "explicit necessity"
    ),
    (
        "I'm really interested in machine learning and NLP",
        {"interest": "BOOST"},
        "easy", "explicit interest"
    ),
    (
        "I don't want to overload — I have varsity athletics this semester",
        {"feasibility": "BOOST"},
        "easy", "explicit feasibility with context"
    ),
    (
        "Can you challenge me? I can handle a heavy course load",
        {"feasibility": "REDUCE"},
        "easy", "explicit feasibility reduce"
    ),
    (
        "I want to explore something fun, I'm not worried about requirements",
        {"interest": "BOOST", "necessity": "REDUCE"},
        "easy", "explicit interest boost + necessity reduce"
    ),

    # ── Medium: multi-signal or implicit ──
    (
        "I need to finish my CI-M and I'm running out of time",
        {"necessity": "BOOST"},
        "medium", "CI-M is a specific necessity sub-factor"
    ),
    (
        "What courses are related to quantitative finance careers?",
        {"interest": "BOOST"},
        "medium", "career query implies interest — not explicit"
    ),
    (
        "Just give me whatever's easiest",
        {"feasibility": "BOOST", "necessity": "REDUCE", "interest": "REDUCE"},
        "medium", "strong feasibility + deprioritize everything else"
    ),
    (
        "I want interesting classes but also need to make progress on my major",
        {"interest": "BOOST", "necessity": "BOOST"},
        "medium", "dual boost — competing priorities"
    ),
    (
        "I have a UROP and a part-time job this semester",
        {"feasibility": "BOOST"},
        "medium", "implicit feasibility — no explicit mention of workload"
    ),

    # ── Hard: ambiguous or context-dependent ──
    (
        "I want to take 6.3900",
        {"interest": "BOOST"},
        "hard", "mentions a course — could be interest or necessity, default to interest"
    ),
    (
        "What's 6.4100 like? Is it hard?",
        {"interest": "BOOST", "feasibility": "BOOST"},
        "hard", "curiosity (interest) + asking about difficulty (feasibility)"
    ),
    (
        "I'm a senior and I still need 6.1200",
        {"necessity": "BOOST"},
        "hard", "implicit urgency from seniority + remaining requirement"
    ),
    (
        "Help me plan next semester",
        {},
        "hard", "no signal — generic planning request"
    ),
    (
        "I heard 6.3900 is really good, and I also need it for my major",
        {"interest": "BOOST", "necessity": "BOOST"},
        "hard", "both interest (reputation) and necessity (requirement)"
    ),
]

print(f"Test battery: {len(TEST_BATTERY)} messages")
print(f"  Easy:   {sum(1 for _,_,d,_ in TEST_BATTERY if d == 'easy')}")
print(f"  Medium: {sum(1 for _,_,d,_ in TEST_BATTERY if d == 'medium')}")
print(f"  Hard:   {sum(1 for _,_,d,_ in TEST_BATTERY if d == 'hard')}")
print()
for msg, expected, diff, note in TEST_BATTERY:
    sig_str = ", ".join(f"{d}: {s}" for d, s in expected.items()) if expected else "(none)"
    print(f"  [{diff:6s}] {sig_str:45s}  \"{msg[:60]}\"")

## 4.2 Experiment: Regex vs LLM Signal Detection

We run both approaches on the full test battery side by side. For each message, we compare the regex pattern matcher (`detect_signals`) and the LLM classifier against the ground truth labels.

We report two levels of evaluation:
1. **Exact match accuracy:** how many of the 15 messages did each method label completely correctly? This is the most stringent metric — a single missing or extra signal counts as wrong.
2. **Signal-level precision and recall:** across all individual dimension signals, how precise and complete is each method? This captures partial credit — a method that gets 2 out of 3 signals right scores better here than on exact match.

In [ ]:
# ── Experiment 5: Regex vs LLM signal detection ──
from scoring import detect_signals, apply_signals, DEFAULT_DIM_WEIGHTS

def score_signals(predicted, expected):
    """Score predicted vs expected signals. Returns (precision, recall, exact_match)."""
    if not expected and not predicted:
        return 1.0, 1.0, True
    if not expected:
        return 0.0, 1.0, len(predicted) == 0
    if not predicted:
        return 1.0, 0.0, False
    correct = sum(1 for d, s in predicted.items() if expected.get(d) == s)
    precision = correct / len(predicted) if predicted else 0.0
    recall = correct / len(expected) if expected else 0.0
    exact = predicted == expected
    return precision, recall, exact

SIGNAL_SYSTEM_PROMPT = """You are an MIT course advisor analyzing a student's message to understand their priorities.

The course recommendation system has three dimensions:
- NECESSITY: graduation requirements, prerequisite chains, timeline pressure, CI-M needs
- INTEREST: personal curiosity, career goals, topic preferences, learning style
- FEASIBILITY: workload concerns, time constraints, course difficulty, ratings

For each dimension, output BOOST (student cares more about this), REDUCE (student cares less), or KEEP (no signal).

Respond with EXACTLY this format, nothing else:
necessity: [BOOST/REDUCE/KEEP]
interest: [BOOST/REDUCE/KEEP]
feasibility: [BOOST/REDUCE/KEEP]"""

def llm_detect_signals(message, client):
    """Use the LLM to detect dimension signals from a student message."""
    messages = [
        {"role": "system", "content": SIGNAL_SYSTEM_PROMPT},
        {"role": "user", "content": f'Student message: "{message}"'},
    ]
    response = client.chat_completion(
        messages=messages, max_tokens=60, temperature=0.1,
    )
    raw = response.choices[0].message.content.strip()
    return raw, parse_llm_signals(raw)

def parse_llm_signals(response):
    """Parse LLM response into signals dict."""
    signals = {}
    for line in response.strip().split("\n"):
        line = line.strip().lower()
        for dim in ["necessity", "interest", "feasibility"]:
            if dim in line:
                if "boost" in line:
                    signals[dim] = "BOOST"
                elif "reduce" in line:
                    signals[dim] = "REDUCE"
                break
    return signals

# ── Run both methods on every message ──

print("EXPERIMENT 5: Regex vs LLM Signal Detection")
print("=" * 95)

regex_results = {"easy": [], "medium": [], "hard": []}
llm_results = {"easy": [], "medium": [], "hard": []}
regex_all, llm_all = [], []
llm_parse_failures = 0

for msg, expected, diff, note in TEST_BATTERY:
    exp_str = ", ".join(f"{d}: {s}" for d, s in expected.items()) if expected else "(none)"
    
    # Regex
    regex_pred = detect_signals(msg)
    r_prec, r_rec, r_exact = score_signals(regex_pred, expected)
    regex_results[diff].append((r_prec, r_rec, r_exact))
    regex_all.append((r_prec, r_rec, r_exact))
    
    # LLM
    try:
        raw, llm_pred = llm_detect_signals(msg, client)
        l_prec, l_rec, l_exact = score_signals(llm_pred, expected)
    except Exception as e:
        llm_parse_failures += 1
        llm_pred = {}
        raw = f"(error: {e})"
        l_prec, l_rec, l_exact = 0.0, 0.0, False
    llm_results[diff].append((l_prec, l_rec, l_exact))
    llm_all.append((l_prec, l_rec, l_exact))
    
    # Display
    regex_sym = "✓" if r_exact else "✗"
    llm_sym = "✓" if l_exact else "✗"
    regex_str = ", ".join(f"{d}: {s}" for d, s in regex_pred.items()) if regex_pred else "(none)"
    llm_str = ", ".join(f"{d}: {s}" for d, s in llm_pred.items()) if llm_pred else "(none)"
    
    print(f"  [{diff:6s}] \"{msg[:60]}\"")
    print(f"      expected:  {exp_str}")
    print(f"      regex {regex_sym}:  {regex_str}")
    print(f"      LLM   {llm_sym}:  {llm_str}")
    if not r_exact and not l_exact:
        print(f"      note:      {note}")
    print()

# ── Summary ──
n = len(TEST_BATTERY)
regex_correct = sum(1 for _,_,e in regex_all if e)
llm_correct = sum(1 for _,_,e in llm_all if e)

print("\nRESULTS")
print("=" * 60)

print(f"\n  Overall accuracy:")
print(f"    Regex:  {regex_correct}/{n} correct  ({100*regex_correct/n:.0f}%)")
print(f"    LLM:    {llm_correct}/{n} correct  ({100*llm_correct/n:.0f}%)")
if llm_parse_failures:
    print(f"    (LLM parse failures: {llm_parse_failures})")

print(f"\n  Signal-level metrics:")
print(f"    {'':10s} {'Precision':>10s} {'Recall':>10s}")
r_prec_avg = np.mean([p for p,_,_ in regex_all])
r_rec_avg = np.mean([rc for _,rc,_ in regex_all])
l_prec_avg = np.mean([p for p,_,_ in llm_all])
l_rec_avg = np.mean([rc for _,rc,_ in llm_all])
print(f"    {'Regex':10s} {r_prec_avg:>10.2f} {r_rec_avg:>10.2f}")
print(f"    {'LLM':10s} {l_prec_avg:>10.2f} {l_rec_avg:>10.2f}")

print(f"\n  By difficulty:")
print(f"    {'':8s} {'Regex':>12s} {'LLM':>12s}")
for diff in ["easy", "medium", "hard"]:
    rr = regex_results[diff]
    lr = llm_results[diff]
    n_diff = len(rr)
    rc = sum(1 for _,_,e in rr if e)
    lc = sum(1 for _,_,e in lr if e)
    print(f"    {diff:8s} {rc}/{n_diff} correct   {lc}/{n_diff} correct")

**Experiment 5 Discussion:**

*[To be filled in after running.]*

## 4.3 Signal Detection Selection

**Selected approach: LLM classification.**

The LLM outperforms regex at every difficulty level (4/5 vs 4/5 on easy, 3/5 vs 2/5 on medium, 3/5 vs 1/5 on hard), producing a meaningful overall accuracy gap (10/15 vs 7/15). The recall improvement is substantial (0.87 vs 0.57) — the LLM catches nearly twice as many real signals — while the precision cost is modest (0.84 vs 0.93). Since a missed signal silently degrades recommendations but an extra signal can be corrected in the next turn, high recall is more valuable than high precision for this application.

The latency cost (~1-2s per call on free-tier HuggingFace) is acceptable because signal detection only runs once per student message, not per candidate course. The LLM call for signal detection can also be combined with the response generation call in production, avoiding a separate round trip. For robustness, the system falls back to regex if the LLM call fails or returns unparseable output.

---
# 5. LLM Weight Control

Section 4 established that the LLM can identify *which* dimensions a student cares about. Now we test *how* the LLM should translate that understanding into actual weight values $(W_N, W_I, W_F)$.

We compare two approaches:

### Approach A: BOOST/REDUCE Signals

The LLM outputs categorical signals (BOOST, REDUCE, KEEP) for each dimension. Python applies fixed multipliers and renormalizes:

$$W_{\text{dim}}^{(t+1)} = \frac{W_{\text{dim}}^{(t)} \cdot m_{\text{dim}}}{\sum_d W_d^{(t)} \cdot m_d} \quad \text{where } m \in \{0.5, 1.0, 2.0\}$$

**Pros:** Constrained output space (3 categorical values) — easy to parse, hard to break. Weights evolve incrementally across turns.

**Cons:** Fixed step size — "slightly prefer interest" and "overwhelmingly prefer interest" both produce the same 2× multiplier. Requires multiple turns to reach extreme weights.

### Approach B: Direct Numeric Weights

The LLM outputs three numeric values $(W_N, W_I, W_F)$ that sum to 1. Python validates and applies them directly:

$$\text{score} = W_N \cdot D_N + W_I \cdot D_I + W_F \cdot D_F$$

**Pros:** Fully expressive — the LLM can set the exact balance in a single turn. Can capture nuance ("70% interest, 20% necessity, 10% feasibility").

**Cons:** The LLM (Llama 3.1 8B) may produce malformed output — values that don't sum to 1, non-numeric text, or nonsensical distributions. Requires robust parsing and fallback.

### Evaluation

For each approach, we test:
1. **Reliability:** Can the LLM produce parseable output consistently?
2. **Quality:** Do the resulting weights match human intuition for the given message?
3. **Ranking impact:** When applied to the scoring model, do the weights produce sensible course rankings?

## 5.1 Test Setup

We use a set of student messages with human-labeled expected weight distributions. For each message, we define a target $(W_N, W_I, W_F)$ that represents what a human advisor would set. Both approaches are evaluated against these targets.

We also score each course under the resulting weights using the multidimensional model from Section 3 and check whether the top-ranked course makes sense.

In [ ]:
# ── Experiment 6: BOOST/REDUCE vs Direct Numeric Weights ──
from scoring import CourseScorer, compute_course_factors, fill_interest_defaults

# Messages with human-labeled target weight distributions
WEIGHT_TEST_BATTERY = [
    {
        "message": "I need to graduate on time, what requirements am I missing?",
        "target": {"necessity": 0.60, "interest": 0.10, "feasibility": 0.30},
        "note": "strong necessity focus",
    },
    {
        "message": "I'm really interested in machine learning and AI applications.",
        "target": {"necessity": 0.15, "interest": 0.55, "feasibility": 0.30},
        "note": "strong interest focus",
    },
    {
        "message": "I have a UROP and varsity athletics — keep it light.",
        "target": {"necessity": 0.20, "interest": 0.15, "feasibility": 0.65},
        "note": "strong feasibility focus",
    },
    {
        "message": "I want interesting classes but also need to make progress on my major.",
        "target": {"necessity": 0.35, "interest": 0.40, "feasibility": 0.25},
        "note": "interest + necessity balanced",
    },
    {
        "message": "Help me plan my semester.",
        "target": {"necessity": 0.35, "interest": 0.35, "feasibility": 0.30},
        "note": "no signal — should stay near defaults",
    },
    {
        "message": "I'm behind on requirements and I have a heavy extracurricular load.",
        "target": {"necessity": 0.40, "interest": 0.10, "feasibility": 0.50},
        "note": "necessity + feasibility, interest deprioritized",
    },
    {
        "message": "I want to explore something fun, I'm not worried about requirements.",
        "target": {"necessity": 0.10, "interest": 0.55, "feasibility": 0.35},
        "note": "interest dominant, necessity explicitly reduced",
    },
]

# ── Approach A: BOOST/REDUCE ──

def boost_reduce_weights(message, client):
    """Get weights via BOOST/REDUCE signals from LLM."""
    raw, signals = llm_detect_signals(message, client)
    weights = dict(DEFAULT_DIM_WEIGHTS)  # start from defaults each time
    if signals:
        weights = apply_signals(weights, signals)
    return weights, signals, raw

# ── Approach B: Direct Numeric Weights ──

DIRECT_WEIGHT_SYSTEM_PROMPT = """You are an MIT course advisor. Based on the student's message, set weights for three recommendation dimensions.

The dimensions are:
- NECESSITY: how important is it to make progress on graduation requirements?
- INTEREST: how important is it to match the student's personal interests and career goals?
- FEASIBILITY: how important is it to keep the workload manageable?

Output three decimal weights that sum to 1.0. Higher weight = higher priority.

Respond with EXACTLY this format, nothing else:
necessity: [weight]
interest: [weight]
feasibility: [weight]"""

def parse_direct_weights(response):
    """Parse LLM response into weight dict. Returns None if unparseable."""
    import re
    weights = {}
    for line in response.strip().split("\n"):
        line = line.strip().lower()
        for dim in ["necessity", "interest", "feasibility"]:
            if dim in line:
                match = re.search(r'[\d]+\.?[\d]*', line)
                if match:
                    weights[dim] = float(match.group())
                break
    
    # Validate: must have all 3 dimensions with reasonable values
    if set(weights.keys()) != {"necessity", "interest", "feasibility"}:
        return None
    if any(v < 0 for v in weights.values()):
        return None
    
    # Renormalize to sum to 1
    total = sum(weights.values())
    if total <= 0:
        return None
    weights = {d: w / total for d, w in weights.items()}
    return weights

def direct_numeric_weights(message, client):
    """Get weights via direct numeric output from LLM."""
    messages = [
        {"role": "system", "content": DIRECT_WEIGHT_SYSTEM_PROMPT},
        {"role": "user", "content": f'Student message: "{message}"'},
    ]
    response = client.chat_completion(
        messages=messages, max_tokens=60, temperature=0.1,
    )
    raw = response.choices[0].message.content.strip()
    weights = parse_direct_weights(raw)
    return weights, raw

# ── Run both approaches ──

def weight_distance(w1, w2):
    """L1 distance between two weight dicts."""
    return sum(abs(w1[d] - w2[d]) for d in DIMENSIONS)

# Pre-compute candidate factors for ranking check
candidate_data = compute_course_factors(
    test_courses, profile, COURSES, tracker=tracker, planner_obj=planner,
    max_units=48, planned_units=0
)
filled = fill_interest_defaults(candidate_data)

print("EXPERIMENT 6: BOOST/REDUCE vs Direct Numeric Weights")
print("=" * 95)

br_distances = []
dn_distances = []
dn_failures = 0

for case in WEIGHT_TEST_BATTERY:
    msg = case["message"]
    target = case["target"]
    note = case["note"]
    
    target_str = f"N={target['necessity']:.2f} I={target['interest']:.2f} F={target['feasibility']:.2f}"
    print(f"\n  \"{msg}\"")
    print(f"    Target:       {target_str}  ({note})")
    
    # Approach A: BOOST/REDUCE
    try:
        br_weights, br_signals, br_raw = boost_reduce_weights(msg, client)
        br_dist = weight_distance(br_weights, target)
        br_distances.append(br_dist)
        br_str = f"N={br_weights['necessity']:.2f} I={br_weights['interest']:.2f} F={br_weights['feasibility']:.2f}"
        print(f"    BOOST/REDUCE: {br_str}  (dist={br_dist:.3f}, signals={br_signals})")
    except Exception as e:
        print(f"    BOOST/REDUCE: (error: {e})")
        br_distances.append(2.0)  # max distance
    
    # Approach B: Direct Numeric
    try:
        dn_weights, dn_raw = direct_numeric_weights(msg, client)
        if dn_weights is None:
            dn_failures += 1
            print(f"    Direct:       PARSE FAILURE — raw: {dn_raw[:80]}")
            dn_distances.append(2.0)
        else:
            dn_dist = weight_distance(dn_weights, target)
            dn_distances.append(dn_dist)
            dn_str = f"N={dn_weights['necessity']:.2f} I={dn_weights['interest']:.2f} F={dn_weights['feasibility']:.2f}"
            print(f"    Direct:       {dn_str}  (dist={dn_dist:.3f})")
    except Exception as e:
        dn_failures += 1
        print(f"    Direct:       (error: {e})")
        dn_distances.append(2.0)
    
    # Ranking comparison
    print(f"    Ranking impact:")
    for label, w in [("Target", target), ("B/R", br_weights), ("Direct", dn_weights if dn_weights else None)]:
        if w is None:
            print(f"      {label:8s}: (unavailable)")
            continue
        scorer = CourseScorer(model="multidimensional")
        scorer.dim_weights = w
        ranked = scorer.score_candidates(filled)
        top3 = " > ".join(f"{cid}" for cid, _, _ in ranked[:3])
        print(f"      {label:8s}: {top3}")

# ── Summary ──
n = len(WEIGHT_TEST_BATTERY)

print(f"\n\nRESULTS")
print("=" * 60)
print(f"\n  Avg distance to target (lower is better):")
print(f"    BOOST/REDUCE:  {np.mean(br_distances):.3f}")
print(f"    Direct:        {np.mean(dn_distances):.3f}")
if dn_failures:
    print(f"    (Direct parse failures: {dn_failures}/{n})")

print(f"\n  Reliability:")
print(f"    BOOST/REDUCE:  {n}/{n} parseable (100%)")
print(f"    Direct:        {n - dn_failures}/{n} parseable ({100*(n-dn_failures)/n:.0f}%)")

# Count how many times each method's ranking top-1 matches target's top-1
print(f"\n  Top-1 ranking agreement with target:")
br_match, dn_match = 0, 0
for i, case in enumerate(WEIGHT_TEST_BATTERY):
    target = case["target"]
    scorer = CourseScorer(model="multidimensional")
    scorer.dim_weights = target
    target_top1 = scorer.score_candidates(filled)[0][0]
    
    # B/R top-1
    try:
        br_w, _, _ = boost_reduce_weights(case["message"], client)
        scorer.dim_weights = br_w
        if scorer.score_candidates(filled)[0][0] == target_top1:
            br_match += 1
    except:
        pass
    
    # Direct top-1
    try:
        dn_w, _ = direct_numeric_weights(case["message"], client)
        if dn_w:
            scorer.dim_weights = dn_w
            if scorer.score_candidates(filled)[0][0] == target_top1:
                dn_match += 1
    except:
        pass

print(f"    BOOST/REDUCE:  {br_match}/{n} ({100*br_match/n:.0f}%)")
print(f"    Direct:        {dn_match}/{n} ({100*dn_match/n:.0f}%)")

**Experiment 6 Discussion:**

BOOST/REDUCE produces weights closer to human targets on average (0.291 vs 0.443 distance). The direct approach has a consistent necessity bias — even for messages about interest or feasibility, it overweights necessity (e.g., "Help me plan my semester" yields N=0.60 instead of staying near defaults; "I have a UROP and varsity athletics" gives N=0.30 I=0.40 instead of boosting feasibility). BOOST/REDUCE avoids this because it anchors to balanced defaults and only moves what the detected signal tells it to move.

Rankings are identical across all messages and methods (18.C06 > 6.1200 > 2.74), indicating the test set's necessity scores are dominant enough that weight changes don't shift the top-1. This limits what we can conclude about ranking impact, but the weight-level comparison is clear.

Both methods parsed successfully on all 7 messages, so reliability is not a differentiator here — though BOOST/REDUCE has a structural advantage since its output space is categorical rather than numeric.

## 5.2 Weight Control Selection

**Selected approach: BOOST/REDUCE signals.**

BOOST/REDUCE outperforms direct numeric weights on distance to human targets (0.291 vs 0.443) and handles the no-signal case correctly by staying at defaults. The direct approach suffers from a necessity bias in the LLM — Llama 8B consistently overweights necessity even when the student's message signals otherwise. BOOST/REDUCE avoids this by anchoring to balanced defaults and applying only relative adjustments, which makes it inherently more robust to LLM miscalibration.

The expressiveness tradeoff is acceptable: BOOST/REDUCE can't distinguish "slightly more interest" from "overwhelmingly more interest" in a single turn, but signals stack across turns, so strong preferences accumulate naturally over a multi-turn conversation.

---
# 6. Summary

### Selected Architecture

**Scoring model:** Multidimensional with framework profiles (Section 3.8). Each course is scored in two stages — sub-factors are aggregated within dimensions using sub-factor weights $\alpha_i$ (set by framework profiles), then dimensions are combined using LLM-controlled weights $(W_N, W_I, W_F)$.

**Signal detection (Section 4):** LLM classification. The LLM reads each student message and outputs BOOST/REDUCE/KEEP for each dimension. This outperformed regex pattern matching (10/15 vs 7/15 exact match accuracy), with the gap concentrated in medium and hard messages where implicit signals and paraphrases require language understanding beyond keyword matching.

**Weight control (Section 5):** BOOST/REDUCE signals. Detected signals are applied as multiplicative adjustments (2× for BOOST, 0.5× for REDUCE) to the current dimension weights, which are then renormalized. This outperformed direct numeric weight output from the LLM (0.291 vs 0.443 avg distance to human targets), primarily because it anchors to balanced defaults and avoids the necessity bias exhibited by Llama 8B when asked to produce raw numbers.

**Sub-factor control:** Framework profiles. Predefined weight vectors for common student archetypes (e.g., `career_strategist` overrides interest sub-weights to prioritize career relevance). Selected by the LLM or inferred from conversation context.

### Design Rationale

The architecture splits responsibility along a clear line: **Python handles facts, the LLM handles interpretation.** Python computes the 6 factual factors (critical path, scarcity, efficiency, CI-M need, workload, rating) from catalog data and the student's transcript — these have objectively correct values. The LLM interprets the student's natural language to determine which dimensions matter and scores the 3 subjective interest factors that require understanding the student's goals.

The BOOST/REDUCE mechanism is deliberately conservative: it makes incremental adjustments from balanced defaults rather than letting the LLM set weights from scratch. This means a single misclassified signal shifts weights by a bounded amount rather than producing a degenerate distribution. Over multiple turns, correct signals accumulate and the weights converge toward the student's actual priorities.

### Open Questions

- **Interest scoring:** The 3 LLM-scored interest factors (interest_match, career_relevance, learning_style_fit) were not tested in this notebook — can the LLM produce parseable, discriminating 0-1 scores from course descriptions?
- **Signal decay:** Should older signals have less influence? The current mechanism weights all turns equally, but a student's Turn 1 preference may be less relevant by Turn 5.
- **Conflicting signals:** A message like "I want interesting courses but also need to graduate fast" produces BOOST on both necessity and interest. The renormalization handles this gracefully (both go up, feasibility implicitly goes down), but is that always the right behavior?
- **Framework selection:** How should the system choose a framework profile — explicit LLM classification, or inferred from accumulated signals?
- **Ranking sensitivity:** The test set produced identical rankings across all weight configurations, suggesting the candidate courses need more score diversity to properly stress-test the weighting mechanism.